# CS336 — lecture_02

Converted from the original Stanford executable lecture.

In [41]:
import os
import regex
import tiktoken
from abc import ABC
from dataclasses import dataclass
from collections import defaultdict

import math
import torch.nn.functional as F
import timeit
from typing import Iterable
import torch
from torch import nn
from einops import rearrange, einsum, reduce

from gpu_util import cuda_if_available, get_max_memory_usage
from facts import h100_flop_per_sec, h100_bytes_per_sec


## Definitions

Classes and helper functions used by the following examples.

In [2]:
class Block(nn.Module):
    """Simple block that applies a linear transformation followed by a ReLU nonlinearity."""
    def __init__(self, dim: int):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(dim, dim) / math.sqrt(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x @ self.weight  # Linear
        x = F.relu(x)        # Activation
        return x


class DeepNetwork(nn.Module):
    """Map `dim`-vector to a `dim`-vector."""
    def __init__(self, dim: int, num_layers: int):
        super().__init__()
        self.layers = nn.ModuleList([Block(dim) for i in range(num_layers)])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Apply all the layers sequentially
        for layer in self.layers:
            x = layer(x)  # @stepover
        return x


class AdaGrad(torch.optim.Optimizer):
    def __init__(self, params: Iterable[nn.Parameter], lr: float = 0.01):
        super(AdaGrad, self).__init__(params, dict(lr=lr))

    def step(self):
        for group in self.param_groups:
            lr = group["lr"]
            for p in group["params"]:
                # Optimizer state
                state = self.state[p]
                grad = p.grad.data

                # Get squared gradients g2 = sum_{i<t} g_i^2
                g2 = state.get("g2", torch.zeros_like(grad))

                # Update optimizer state
                g2 += torch.square(grad)
                state["g2"] = g2

                # Update parameters
                p.data -= lr * grad / torch.sqrt(g2 + 1e-5)


class DeepNetworkCheckpointed(nn.Module):
    """Same as DeepNetwork, but with activation checkpointing."""
    def __init__(self, dim: int, num_layers: int):
        super().__init__()
        self.layers = nn.ModuleList([Block(dim) for i in range(num_layers)])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Apply all the layers sequentially
        for layer in self.layers:
            # KEY: only store activations at checkpoints, recompute the rest
            x = torch.utils.checkpoint.checkpoint(layer, x)  # @stepover
        return x


def get_memory_usage(x: torch.Tensor):
    return x.numel() * x.element_size()


def benchmark(func, num_trials: int = 5) -> float:
    """Return the number of seconds required to perform `func`."""

    # Wait until previous CUDA threads are done
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    def run():
        # Perform the operation
        func()

        # Wait until CUDA threads are done
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    # Time the operation `num_trials` times
    total_time = timeit.timeit(run, number=num_trials)

    return total_time / num_trials


Announcements:

- Join the CS336 slack

- Sign up on Modal with your **Stanford** email

- Read the [AI policy guide](https://docs.google.com/document/d/1SZAlExB1qAc9izHt54gwunNpjKE6wXb8Y7yA_e-baK8/edit?tab=t.0)

- Read the [cluster guide](https://docs.google.com/document/d/1cHE0iKVyXLJ3XpIs2XuXTmZ-HMmPk2hIPeCvy-AydMg/edit?tab=t.otis27tacaef)

Marin 1e23 FLOPs run finished and [matched forecasts](https://x.com/WilliamBarrHeld/status/2039373983632814318)!

![Lecture image](https://pbs.twimg.com/media/HE1P1HmaUAAjLXF?format=jpg&name=medium)

Last lecture: overview, tokenization

Today: resource accounting (systems)

Recall: what's the best model one can train given fixed resources (compute, memory)?

In other words: maximize (computational) **efficiency**.

Prerequisite: understand the resources (compute, memory) for a given computation.

**Question**: How long would it take to train a 70B parameter model on 15T tokens on 1024 H100s?

In [40]:
total_flops = 6 * 70e9 * 15e12  # @inspect total_flops

h100_flop_per_sec = 1979e12 / 2

mfu = 0.5

flops_per_day = h100_flop_per_sec * mfu * 1024 * 60 * 60 * 24  # @inspect flops_per_day

days = total_flops / flops_per_day  # @inspect days

**Question**: What's the largest model that can you can train on 8 H100s using AdamW?

In [4]:
h100_bytes = 80e9  # @inspect h100_bytes

bytes_per_parameter = 2 + 2 + (4 + 4)  # parameters (2), gradients (2), optimizer state (4 + 4) @inspect bytes_per_parameter

num_parameters = (h100_bytes * 8) / bytes_per_parameter  # @inspect num_parameters

Caveat: activations are not accounted for (depends on batch size and sequence length), so this is an upper bound.

This is a rough back-of-the-envelope calculation.

But it gives you the flavor of napkin math one can quickly do to get a sense of resources.

What knowledge to take away from this lecture:

- Mechanics: straightforward (PyTorch semantics)

- Mindset: resource accounting (remember to do it)

- Intuitions: get a sense of how resources are spent, no ML magic today

Tensors are the basic building block for storing everything:

- data

- parameters

- gradients

- optimizer state

- activations

Example: parameters of the DeepSeek v3.2 model  *[deepseek_v3_2_2025]*

 [DeepSeek v3.2 model on Hugging Face](https://huggingface.co/deepseek-ai/DeepSeek-V3.2?show_file_info=model.safetensors.index.json)

Each tensor has a rank, which is the number of dimensions.

In [5]:
x = torch.zeros(4)        # rank 1 tensor (vector) @inspect x

x = torch.zeros(4, 8)     # rank 2 tensor (matrix) @inspect x

x = torch.zeros(4, 8, 2)  # rank 3 tensor @inspect x

In Transformers, will see tensors of rank 4:

In [6]:
B = 32   # Batch size

S = 16   # Sequence length

H = 16   # Number of heads

D = 64   # Hidden dimension per head

x = torch.zeros(B, S, H, D)

Elements of tensors are generally floating point numbers.

## fp32

 [Wikipedia](https://en.wikipedia.org/wiki/Single-precision_floating-point_format)

![Lecture image](images/fp32.png)

The fp32 data type (also known as float32 or single precision) is the default.

Traditionally, in scientific computing, fp32 is the baseline; you could use double precision (fp64) in some cases.

In deep learning, you can be a lot sloppier.

Let's examine memory usage of these tensors.

Memory is determined by the (i) number of values and (ii) data type of each value.

In [7]:
x = torch.zeros(4, 8)  # @inspect x

assert x.dtype == torch.float32  # Default type

assert x.numel() == 4 * 8

assert x.element_size() == 4  # Float is 4 bytes

assert get_memory_usage(x) == 4 * 8 * 4  # 128 bytes

One matrix in the feedforward layer of GPT-3:

In [8]:
assert get_memory_usage(torch.empty(12288 * 4, 12288)) == 2304 * 1024 * 1024  # 2.3 GB @stepover

## fp16

 [Wikipedia](https://en.wikipedia.org/wiki/Half-precision_floating-point_format)

![Lecture image](images/fp16.png)

The fp16 data type (also known as float16 or half precision) cuts down the memory.

In [9]:
x = torch.zeros(4, 8, dtype=torch.float16)  # @inspect x

assert x.element_size() == 2

However, the dynamic range (especially for small numbers) isn't great.

In [10]:
x = torch.tensor([1e-8], dtype=torch.float16)  # @inspect x

assert x == 0  # Underflow!

If this happens when you train, you can get instability.

## bf16

 [Wikipedia](https://en.wikipedia.org/wiki/Bfloat16_floating-point_format)

![Lecture image](images/bf16.png)

Google Brain developed brain floating point (bf16) in 2018 to address this issue.

bf16 uses the same memory as fp16 but has the same dynamic range as fp32!

The only catch is that the resolution is worse, but this matters less for deep learning.

In [11]:
x = torch.tensor([1e-8], dtype=torch.bfloat16)  # @inspect x

assert x != 0  # No underflow!

## Mixed precision

Implications on training:

- Training with fp32 works, but requires lots of memory.

- Training with fp16 and even bf16 is risky, and you can get instability.

Solution: mixed precision training  [https://arxiv.org/pdf/1710.03740.pdf](https://arxiv.org/pdf/1710.03740.pdf)

- Use bf16 for parameters, activations, and gradients

- Use fp32 for optimizer states

Pytorch has an automatic mixed precision (AMP) library.  [docs](https://pytorch.org/docs/stable/amp.html)

Tries to cast things into bf16 when safe (matmuls, not exp).

In [12]:
with torch.amp.autocast("cuda", dtype=torch.bfloat16):
    x = torch.zeros(4, 8)  # @inspect x

## fp8

In 2022, fp8 was standardized, motivated by machine learning workloads [primer](https://docs.nvidia.com/deeplearning/transformer-engine/user-guide/examples/fp8_primer.html).

![Lecture image](https://docs.nvidia.com/deeplearning/transformer-engine/user-guide/_images/fp8_formats.png)

H100s support two variants of FP8: E4M3 (range [-448, 448]) and E5M2 ([-57344, 57344]).

Reference:  [https://arxiv.org/pdf/2209.05433.pdf](https://arxiv.org/pdf/2209.05433.pdf)

## fp4

In 2025, NVIDIA developed [nvfp4](https://developer.nvidia.com/blog/introducing-nvfp4-for-efficient-and-accurate-low-precision-inference/)

Only 4 bits per value!

Values: -6, -4, -3, -2, -1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2, 3, 4, 6

Use a separate scale factor per block, so actually get more dynamic range (but just can't vary freely from neighbors).

Nemotron 3 Super was trained in NVFP4  *[nemotron_3_super_2026]*

Some of this is done in NVIDIA libraries outside of user control.

By default, tensors are stored in CPU memory.

In [13]:
x = torch.zeros(32, 32)

assert x.device == torch.device("cpu")

However, what about GPUs?

![Lecture image](images/cpu-gpu.png)

In [14]:
device = cuda_if_available()  # @inspect device

In order to take advantage of the massive parallelism of GPUs, we need to move them to GPU memory.

In [15]:
x = x.to(device)

Or create the tensor directly on the GPU:

In [16]:
with torch.device(device):
    x = torch.zeros(32, 32)
    assert x.device == device

Traditional PyTorch code:

In [17]:
x = torch.ones(2, 2, 3)      # batch seq hidden  @inspect x

y = torch.ones(2, 2, 3)      # batch seq hidden  @inspect y

z = x @ y.transpose(-2, -1)  # batch seq seq  @inspect z

Easy to mess up the dimensions (what is -2, -1?)...

Einops is a library for manipulating tensors where dimensions are named.

It is inspired by Einstein summation notation (Einstein, 1916).

 [Einops tutorial](https://einops.rocks/1-einops-basics/)

Einsum is generalized matrix multiplication with good bookkeeping.

In [18]:
x = torch.ones(3, 4)  # seq1 hidden @inspect x

y = torch.ones(4, 3)  # hidden seq2 @inspect y

z = x @ y   # seq1 seq2 @inspect z

z = einsum(x, y, "seq1 hidden, hidden seq2 -> seq1 seq2")  # @inspect z

z.shape

torch.Size([3, 3])

Let's try a more complex example...

In [19]:
x = torch.ones(2, 3, 4)  # batch seq1 hidden @inspect x

y = torch.ones(2, 3, 4)  # batch seq2 hidden @inspect y

z = x @ y.transpose(-2, -1)  # batch seq1 seq2  @inspect z

z = einsum(x, y, "batch seq1 hidden, batch seq2 hidden -> batch seq1 seq2")  # @inspect z

z.shape

torch.Size([2, 3, 3])

Dimensions that are not named in the output are summed over.

In [20]:
z = einsum(x, y, "... seq1 hidden, ... seq2 hidden -> ... seq1 seq2")  # @inspect z

You can reduce a single tensor via some operation (e.g., sum, mean, max, min).

In [21]:
x = torch.ones(2, 3, 4)  # batch seq hidden @inspect x

y = x.sum(dim=-1)  # @inspect y

y = reduce(x, "... hidden -> ...", "sum")  # @inspect y

Sometimes, a dimension represents two dimensions

...and you want to operate on one of them.

In [22]:
x = torch.ones(3, 8)  # seq total_hidden @inspect x

...where `total_hidden` is a flattened representation of `heads * hidden1`

In [23]:
w = torch.ones(4, 4)  # hidden1 hidden2 @inspect w

x = rearrange(x, "... (heads hidden1) -> ... heads hidden1", heads=2)  # @inspect x

x = einsum(x, w, "... hidden1, hidden1 hidden2 -> ... hidden2")  # @inspect x

x = rearrange(x, "... heads hidden2 -> ... (heads hidden2)")  # @inspect x

Having gone through all the operations, let us examine their computational cost.

A floating-point operation (FLOP) is a basic operation like addition (x + y) or multiplication (x y).

Two terribly confusing acronyms (pronounced the same!):

- FLOPs: floating-point operations (measure of computation done)

- FLOP/s: floating-point operations per second (also written as FLOPS), which is used to measure the speed of hardware.

## Intuitions

Training GPT-3 (2020) took 3.14e23 FLOPs.  [https://lambdalabs.com/blog/demystifying-gpt-3](https://lambdalabs.com/blog/demystifying-gpt-3)

Training GPT-4 (2023) is speculated to take 2e25 FLOPs.  [https://patmcguinness.substack.com/p/gpt-4-details-revealed](https://patmcguinness.substack.com/p/gpt-4-details-revealed)

H100 has a peak performance of 1979 teraFLOP/s with sparsity, 50% without  [spec](https://resources.nvidia.com/en-us-tensor-core/nvidia-tensor-core-gpu-datasheet)

In [24]:
h100_flop_per_sec = 1979e12 / 2

8 H100s for 2 weeks:

In [25]:
total_flops = 8 * 2 * (60 * 60 * 24 * 7) * h100_flop_per_sec  # @inspect total_flops
total_flops

9.5751936e+21

## Linear model

In [26]:
if torch.cuda.is_available():
    B = 16384  # Number of points
    D = 32768  # Dimension of each point
    K = 8192   # Number of outputs
else:
    B = 10240
    D = 2560
    K = 640

x = torch.ones(B, D, device=cuda_if_available())

w = torch.randn(D, K, device=cuda_if_available())

y = x @ w

How many FLOPs is this matmul?

We have one multiplication (x[i][j] * w[j][k]) and one addition per (i, j, k) triple.

In [27]:
actual_num_flops = 2 * B * D * K  # @inspect actual_num_flops
actual_num_flops

8796093022208

We can also time this operation to see how long it takes.

In [29]:
actual_time = benchmark(lambda: x @ w,num_trials=10)  # @inspect actual_time
actual_time

0.6318035998963751

The actual FLOP/s of this operation:

In [30]:
actual_flop_per_sec = actual_num_flops / actual_time  # @inspect actual_flop_per_sec
actual_flop_per_sec

13922195162627.572

Each GPU has a specification sheet that provides the peak performance.

- Example:  [H100 spec](https://resources.nvidia.com/en-us-gpu-resources/h100-datasheet-24306)

Note that the FLOP/s depends heavily on the data type!

In [32]:
def get_promised_flop_per_sec(dtype: torch.dtype) -> float:
    """Return the peak FLOP/s for `device` operating on `dtype`."""
    if not torch.cuda.is_available():
        # No CUDA device available, so can't get FLOP/s
        return 1
    properties = torch.cuda.get_device_properties(cuda_if_available())  # @inspect properties.name

    if "A100" in properties.name:
        # https://www.nvidia.com/content/dam/en-zz/Solutions/Data-Center/a100/pdf/nvidia-a100-datasheet-us-nvidia-1758950-r4-web.pdf
        if dtype == torch.float32:
            return 19.5e12
        if dtype in (torch.bfloat16, torch.float16):
            return 312e12
        raise ValueError(f"Unknown dtype: {dtype}")

    if "H100" in properties.name:
        # https://www.nvidia.com/en-us/data-center/h100/
        if dtype == torch.float32:
            return 67.5e12
        if dtype in (torch.bfloat16, torch.float16):
            return 1979e12 / 2  # 1979 is for sparse, dense is half of that
        raise ValueError(f"Unknown dtype: {dtype}")

    if "B200" in properties.name:
        # https://www.primeline-solutions.com/media/categories/server/nach-gpu/nvidia-hgx-h200/nvidia-blackwell-b200-datasheet.pdf
        if dtype == torch.float32:
            return 75e12
        if dtype in (torch.bfloat16, torch.float16):
            return 4.5e15 / 2  # 4.5e15 is for sparse, dense is half of that
        raise ValueError(f"Unknown dtype: {dtype}")

    if "V100" in properties.name:
        if dtype == torch.float32:
            return 14e12
        if dtype == torch.float16:
            return 112e12
        if dtype == torch.float64:
            return 7e12
        if dtype == torch.bfloat16:
            raise ValueError("V100 does not support BF16 natively")
        raise ValueError(f"Unknown dtype: {dtype}")

    # Unknown GPU: return None so caller can handle gracefully
    return None


In [33]:
promised_flop_per_sec = get_promised_flop_per_sec(x.dtype)  # @inspect promised_flop_per_sec
promised_flop_per_sec

14000000000000.0

## Model FLOPs utilization (MFU)

Definition: MFU = (actual FLOP/s) / (promised FLOP/s) [ignore communication/overhead]

In [35]:
mfu = actual_flop_per_sec / promised_flop_per_sec if promised_flop_per_sec else None  # @inspect mfu
mfu

0.9944425116162552

Usually, MFU of ≥ 0.5 is quite good!

But why is MFU not closer to 1?

To answer this question, we need to look more closely at how computations are done on GPUs...

![Lecture image](images/compute-memory.png)

How to compute a thing:

1. Send inputs from memory to accelerator

2. Perform computation

3. Send outputs from accelerator to memory

How long does this take?

Depends on two things:

1. Accelerator speed (FLOP/s)

2. Memory bandwidth (bytes/s)

In [42]:
assert h100_flop_per_sec == 1979e12 / 2  # Half without sparsity

assert h100_bytes_per_sec == 3.35e12

In [47]:
print(torch.cuda.is_bf16_supported())
print(torch.cuda.is_bf16_supported(including_emulation=False)) # My V100 do not support bf16 natively

True
False


In [50]:
n = 1024 * 1024

x = torch.ones(n, dtype=torch.bfloat16, device=cuda_if_available())

y = torch.relu(x)

bytes = (2 * n) + (2 * n)  # Read x, write y (bf16 is 2 bytes/float) (i.e 16 bit/float)

flops = n  # n comparisons

communication_time = bytes / h100_bytes_per_sec  # @inspect communication_time
print(communication_time)
computation_time = flops / h100_flop_per_sec  # @inspect computation_time
print(computation_time)


1.2520310447761193e-06
1.0597028802425467e-09


Assume we can overlap communication and computation perfectly. -> pratically we cant perfectly  overlap the two operation but lets immagine that we can

In [52]:
total_time = max(communication_time, computation_time)  # @inspect total_time
total_time

1.2520310447761193e-06

What is the bottleneck?

- Memory-bound: communication time > computation time

- Compute-bound: computation time > communication time

In this case, ReLU is memory-bound.

Alternative way to see this:

Accelerator intensity: how much work can the accelerator do per byte transferred?

In [53]:
h100_accelerator_intensity = h100_flop_per_sec / h100_bytes_per_sec  # @inspect h100_accelerator_intensity
h100_accelerator_intensity

295.3731343283582

Arithmetic intensity: how much actual work per byte for this workload?

In [56]:
arithmetic_intensity = flops / bytes  # ~1/4 @inspect arithmetic_intensity
arithmetic_intensity  

0.25

What is the bottleneck?

- Memory-bound: arithmetic intensity < accelerator intensity

- Compute-bound: arithmetic intensity > accelerator intensity

In [60]:
def inspect_bound(arithmetic_intensity,h100_accelerator_intensity):
    if arithmetic_intensity < h100_accelerator_intensity:
        print('Memory-Bound!')
    if arithmetic_intensity > h100_accelerator_intensity:
        print('Compute-Bound!')

In general, we'll find ourselves memory bound.

Can we increase arithmetic intensity?

In [63]:
n = 1024 * 1024

x = torch.ones(n, dtype=torch.bfloat16, device=cuda_if_available())

y = F.gelu(x)  # GELU(x) = 0.5 x (1 + tanh(sqrt(2/pi) (x + 0.044715 x^3)))

bytes = (2 * n) + (2 * n)  # Read x, write y (bf16 is 2 bytes/float)

flops = 20 * n  # tanh can be approximated in various ways (e.g., polynomials)

arithmetic_intensity = flops / bytes  # @inspect arithmetic_intensity
print('arithmetic_intensity',arithmetic_intensity)

h100_accelerator_intensity = h100_flop_per_sec / h100_bytes_per_sec  # @inspect h100_accelerator_intensity

inspect_bound(arithmetic_intensity, h100_accelerator_intensity)

arithmetic_intensity 5.0
Memory-Bound!


Note that GeLU does more work than ReLU per byte moved, so it has higher arithmetic intensity.

But still memory-bound!

In other words, ReLU is not faster than GeLU (when doing things in an isolated way).

In [64]:
n = 1024 * 1024

x = torch.ones(n, dtype=torch.bfloat16, device=cuda_if_available())

w = torch.ones(n, dtype=torch.bfloat16, device=cuda_if_available())

y = x @ w

bytes = (2 * n) + (2 * n) + 2  # Read x, read w, write y

flops = 2 * n - 1  # n multiplications, n-1 additions

arithmetic_intensity = flops / bytes  # ~1/2 @inspect arithmetic_intensity

h100_accelerator_intensity = h100_flop_per_sec / h100_bytes_per_sec  # @inspect h100_accelerator_intensity

assert arithmetic_intensity < h100_accelerator_intensity

Memory-bound!

In [65]:
n = 1024

x = torch.ones(n, dtype=torch.bfloat16, device=cuda_if_available())

w = torch.ones(n, n, dtype=torch.bfloat16, device=cuda_if_available())

y = x @ w

bytes = (2 * n) + (2 * n * n) + (2 * n)  # Read x, read w, write y

flops = n * (2 * n - 1)  # n dot-products

arithmetic_intensity = flops / bytes  # ~1 @inspect arithmetic_intensity

h100_accelerator_intensity = h100_flop_per_sec / h100_bytes_per_sec  # @inspect h100_accelerator_intensity

assert arithmetic_intensity < h100_accelerator_intensity

Memory-bound!

In [66]:
n = 1024

x = torch.ones(n, n, dtype=torch.bfloat16, device=cuda_if_available())

w = torch.ones(n, n, dtype=torch.bfloat16, device=cuda_if_available())

y = x @ w

bytes = (2 * n * n) + (2 * n * n) + (2 * n * n)  # Read x, read w, write y

flops = n * n * (2 * n - 1)  # n^2 dot products

arithmetic_intensity = flops / bytes  # ~n/3 @inspect arithmetic_intensity
 
h100_accelerator_intensity = h100_flop_per_sec / h100_bytes_per_sec  # @inspect h100_accelerator_intensity

assert arithmetic_intensity > h100_accelerator_intensity

Finally, compute-bound!

As long as we have large matrices, we're compute-bound (saturating the accelerator).

Training Transformers involves big matrix multiplications.

Matrix-vector product is what happens during inference, which is why inference is memory-bound.

Note: arithmetic/accelerator intensity also depends on the precision (bf16 versus fp32).

We can visualize the relationship between arithmetic intensity and performance using roofline plots.

![Lecture image](https://jax-ml.github.io/scaling-book/assets/img/roofline-improved-1400.webp)

- Each slice on the x-axis is a particular computation (with some arithmetic intensity)

- Each piecewise linear function corresponds to a particular hardware

- Kink is the accelerator intensity (transition from memory-bound to compute-bound)

We can now relate this back to MFU:

MFU = min(1, arithmetic-intensity / accelerator-intensity)

 [reference](https://jax-ml.github.io/scaling-book/roofline/)

![Lecture image](images/deep-network.png)

Consider a deep network with L layers and D-dimensional inputs, activations, and outputs.

In [70]:
def get_num_parameters(model: nn.Module) -> int:
    return sum(param.numel() for param in model.parameters())

In [72]:
D = 8  # Dimensionality of input, activations, and output

L = 3  # Number of layers

model = DeepNetwork(dim=D, num_layers=L).to(cuda_if_available())

num_parameters = get_num_parameters(model)  # @inspect num_parameters @stepover
print(num_parameters)
assert num_parameters == (D * D) * L  

B = 4  # Batch size

x = torch.randn(B, D, device=cuda_if_available())  # @inspect x

y = model(x)  # @inspect y

192


So far, we've constructed tensors and passed them through operations (forward).

Now, we're going to compute the gradient (backward).

As a simple example, let's consider the simple linear model:

y = 0.5 (x * w - 5)^2

Forward pass: compute loss

In [84]:
x = torch.tensor([1., 2, 3])

w = torch.tensor([1., 1, 1], requires_grad=True)  # Want gradient

pred_y = x @ w

loss = 0.5 * (pred_y - 5).pow(2)

pred_y

tensor(6., grad_fn=<DotBackward0>)

Backward pass: compute gradients

In [77]:
loss.backward()

assert torch.equal(w.grad, torch.tensor([1, 2, 3]))  # @inspect w.grad
print(w.grad)

tensor([1., 2., 3.])


Let us count the FLOPs for computing gradients.

![Lecture image](images/deep-network.png)

In [85]:
B = 1024  # Number of points

D = 256   # Dimension

Define a simplified model (2-layer linear network):

In [86]:
x = torch.ones(B, D, device=cuda_if_available())

w1 = torch.randn(D, D, device=cuda_if_available(), requires_grad=True)

w2 = torch.randn(D, D, device=cuda_if_available(), requires_grad=True)

h1 = einsum(x, w1, "batch in, in out -> batch out")  # x @ w1

h2 = einsum(h1, w2, "batch in, in out -> batch out")  # h1 @ w2

loss = (h2.mean() - 0)**2  # Regress everything to 0 (arbitrary)

h1.retain_grad()  # For debugging

h2.retain_grad()  # For debugging

loss.backward()

/home/dauin_user/gpaoletti/CS336-LM_From_Scratch/lectures-cs336/.venv/lib/python3.13/site-packages/einops/parsing.py:72: RuntimeWarning: It is discouraged to use axes names that are keywords: in
  is_axis_name, reason = self.check_axis_name_return_reason(x, allow_underscore=allow_underscore)


## Zoom in on one layer

Let's focus on the second layer (h2 = h1 @ w2)

**Forward pass**: Recall the number of forward FLOPs: (Remember that 2* its beacuse it is a multiplication and an addition)

In [ ]:
num_forward_flops = 2 * B * D * D   # @inspect num_forward_flops

**Backward pass**: How many FLOPs is running the backward pass?

We need to compute:

- h1.grad = d loss / d h1

- w2.grad = d loss / d w2

In [ ]:
h1_grad = einsum(h2.grad, w2, "batch out, in out -> batch in")

assert torch.allclose(h1.grad, h1_grad)

w2_grad = einsum(h2.grad, h1, "batch out, batch in -> in out")

assert torch.allclose(w2.grad, w2_grad)

num_backward_flops = (2 * B * D * D) + (2 * B * D * D)  # @inspect num_backward_flops

**Note that the backward pass is 2x more expensive than the forward pass.** beacuse we need to compute two gradients

## Consider all layers

This was just for w2, need to apply it to all parameters in the network.

Putting it together:

- Forward pass: 2 (# data points) (# parameters) FLOPs

- Backward pass: 4 (# data points) (# parameters) FLOPs

- Total: **6** (# data points) (# parameters) FLOPs

This is for multilayer perceptrons (MLPs)

...but it turns out to be a good approximation for Transformers for short context lengths as well.

Recall our deep network.

In [87]:
B = 2  # Batch size

D = 4  # Dimensionality of input, activations, and output

L = 3  # Number of layers

model = DeepNetwork(dim=D, num_layers=L).to(cuda_if_available())  # @stepover

Let's define the AdaGrad optimizer

- momentum = SGD + exponential averaging of grad

- AdaGrad = SGD + averaging by grad^2

- RMSProp = AdaGrad but with exponential averaging of grad^2

- Adam = RMSProp + momentum

AdaGrad  *[adagrad_2011]*

In [88]:
optimizer = AdaGrad(model.parameters(), lr=0.01)  # @stepover

state = model.state_dict()  # @inspect state

x = torch.randn(B, D, device=cuda_if_available())

y = torch.tensor([4., 5.], device=cuda_if_available())

pred_y = model(x).mean()  # @stepover

loss = F.mse_loss(input=pred_y, target=y)

loss.backward()

optimizer.step()

optimizer_state = {i: dict(p_state) for i, (p, p_state) in enumerate(optimizer.state.items())}  # @inspect optimizer_state

optimizer.zero_grad(set_to_none=True)

/tmp/ipykernel_3617096/49438799.py:11: UserWarning: Using a target size (torch.Size([2])) that is different to the input size (torch.Size([])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  loss = F.mse_loss(input=pred_y, target=y)


## Memory

In [89]:
num_parameters = D * D * L

parameter_memory = 2 * num_parameters  # (2 bytes for bf16) @inspect parameter_memory

gradient_memory = 2 * num_parameters  # (2 bytes for bf16) @inspect gradient_memory

optimizer_state_memory = 4 * num_parameters  # (4 bytes for fp32) @inspect optimizer_state_memory

activation_memory = 2 * (B * D * L)  # (2 bytes for bf16) @inspect activation_memory

It is customary to use fp32 for stability (accumulating averages over powers over many steps).

Optimizer state memory:

- AdaGrad: 4 bytes/parameter for storing second moments

- Adam: 8 bytes/parameter for storing first and second moments

In [90]:
total_memory = parameter_memory + activation_memory + gradient_memory + optimizer_state_memory  # @inspect total_memory

## Compute (for one training step)

In [91]:
num_parameters = D * D * L

flops = 6 * B * num_parameters  # @inspect flops

## Transformers

The accounting for a Transformer is more complicated, but the same idea.

Assignment 1 will ask you to do that.

Blog post describing memory usage for Transformer training  [https://erees.dev/transformer-memory/](https://erees.dev/transformer-memory/)

Blog post describing FLOPs for a Transformer:  [https://www.adamcasson.com/posts/transformer-flops](https://www.adamcasson.com/posts/transformer-flops)

In [92]:
D = 16  # Dimensionality

true_w = torch.arange(D, dtype=torch.float32, device=cuda_if_available())

B = 4  # Batch size

def get_batch() -> tuple[torch.Tensor, torch.Tensor]:
    x = torch.randn(B, D).to(cuda_if_available())
    true_y = x @ true_w
    return (x, true_y)

L = 2  # Number of layers

model = DeepNetwork(dim=D, num_layers=L).to(cuda_if_available()) # @stepover

optimizer = AdaGrad(model.parameters(), lr=0.01) # @stepover

num_train_steps = 3

for t in range(num_train_steps):
    # Get data
    x, y = get_batch()

    # Forward (compute loss)
    pred_y = model(x).mean()  # @stepover
    loss = F.mse_loss(pred_y, y)

    # Backward (compute gradients)
    loss.backward()

    # Update parameters
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)

/tmp/ipykernel_3617096/1317171233.py:26: UserWarning: Using a target size (torch.Size([4])) that is different to the input size (torch.Size([])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  loss = F.mse_loss(pred_y, y)


Large batch sizes: improve training stability

However, activation memory scales with batch size, so might run out.

In [93]:
B = 64     # Batch size

D = 1024   # Dimensionality

L = 16     # Number of layers

activation_memory = 2 * B * D * L  # (2 bytes for bf16) @inspect activation_memory

Gradient accumulation:

- Compute gradient on micro batches

- Accumulate the gradients (don't zero it out)

- Every batch_size / micro_batch_size steps, update the parameters and zero out the gradients

In [94]:
micro_batch_size = B / 4

activation_memory = 2 * micro_batch_size * D * L  # (2 bytes for bf16) @inspect activation_memory

For training, we need to store the activations of all layers

For inference, we don't compute gradients, so we only need to store the current layer's activations.

![Lecture image](images/deep-network.png)

The memory usage is

In [95]:
B = 64     # Batch size

D = 1024   # Dimensionality

L = 16     # Number of layers

x = torch.randn(B, D, device=cuda_if_available(), requires_grad=True)

activation_memory = 2 * B * D * L  # @inspect activation_memory

model = DeepNetwork(dim=D, num_layers=L).to(cuda_if_available())  # @stepover

memory = get_max_memory_usage(lambda: model(x).sum().backward())  # @inspect memory @stepover

Can we reduce this?

Activation checkpointing = gradient checkpointing = rematerialization

Key idea:

- Forward pass: keep only activations at subset of layers

- Backward pass: recompute the missing activations from the last checkpoint

Philosophy: tradeoff memory for compute

In [96]:
model = DeepNetworkCheckpointed(dim=D, num_layers=L).to(cuda_if_available())  # @stepover

checkpointed_memory = get_max_memory_usage(lambda: model(x).sum().backward())  # @inspect checkpointed_memory @stepover

/home/dauin_user/gpaoletti/CS336-LM_From_Scratch/lectures-cs336/.venv/lib/python3.13/site-packages/torch/_dynamo/eval_frame.py:1446: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Can we reduce this even more, especially for deep networks (large L)?

How frequently to checkpoint?

- If store each layer's activations, then activation memory is O(L) and no recomputation.

- If store no activations, then activation memory is O(1) and compute is O(L^2) (recompute from the start for each layer).

- If store every sqrt(L) layers, then activation memory is O(sqrt(L)) and O(L) recomputation.

Summary:

- Everything is operations on tensors (parameters, gradients, activations, optimizer states, data)

- einops: better way to think about tensor operations

- 6 (# data points) (# parameters) FLOPs per training step

- Arithmetic intensity / roofline analysis: compute-bound or memory-bound?

- Matrix multiplications are compute-bound, elementwise operations are memory-bound

- Gradient accumulation, activation checkpointing: reduce memory to use bigger batch sizes